#### LOINC Logical Observation Identifiers Names and Codes

This file extracts and analyzes LOINC data; the goal is to find condensations (dimensional reductions of LOINC data.

This file was developed with help from Claude AI (https://claude.ai/chat/f79f7068-8cc2-44e2-b09c-f62b12c82d88, chat name 'LOINC Coding Schemes')

#### The current LOINC file is LOINC 2.83, downloaded on August 23, 2026, and imported from Barry's PC.  

TO DO: download to better storage

In [1]:
#### Retrieve LOINC.csv

import pandas as pd
from google.colab import files
uploaded = files.upload()  # select Loinc.csv from the picker

Saving Loinc.csv to Loinc.csv


In [2]:
import pandas as pd
df = pd.read_csv("Loinc.csv", low_memory=False, dtype=str)
print(df.shape)
df.head(3)

(112405, 40)


,LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,VersionLastChanged,CHNG_TYPE,...,COMMON_TEST_RANK,COMMON_ORDER_RANK,HL7_ATTACHMENT_STRUCTURE,EXTERNAL_COPYRIGHT_LINK,PanelType,AskAtOrderEntry,AssociatedObservations,VersionFirstReleased,ValidHL7AttachmentRequest,DisplayName
0,100000-9,Health informatics pioneer and the father of L...,Hx,Pt,^Patient,Nar,NaN,H&P.HX,2.74,ADD,...,0,0,NaN,NaN,NaN,NaN,NaN,2.74,NaN,NaN
1,100001-7,Health informatics pioneer and cofounder of LOINC,Hx,Pt,^Patient,Nar,NaN,H&P.HX,2.74,ADD,...,0,0,NaN,NaN,NaN,NaN,NaN,2.74,NaN,NaN
2,100002-5,Specimen care is maintained,Find,Pt,^Patient,Ord,NaN,SURVEY.PNDS,2.72,ADD,...,0,0,NaN,NaN,NaN,NaN,NaN,2.72,NaN,NaN


In [10]:
print('DataFrame Columns and Types:')
for col_name, col_type in df.dtypes.items():
    print(f'Column: {col_name}, Type: {col_type}')

DataFrame Columns and Types:
Column: LOINC_NUM, Type: object
Column: COMPONENT, Type: object
Column: PROPERTY, Type: object
Column: TIME_ASPCT, Type: object
Column: SYSTEM, Type: object
Column: SCALE_TYP, Type: object
Column: METHOD_TYP, Type: object
Column: CLASS, Type: object
Column: VersionLastChanged, Type: object
Column: CHNG_TYPE, Type: object
Column: DefinitionDescription, Type: object
Column: STATUS, Type: object
Column: CONSUMER_NAME, Type: object
Column: CLASSTYPE, Type: object
Column: FORMULA, Type: object
Column: EXMPL_ANSWERS, Type: object
Column: SURVEY_QUEST_TEXT, Type: object
Column: SURVEY_QUEST_SRC, Type: object
Column: UNITSREQUIRED, Type: object
Column: RELATEDNAMES2, Type: object
Column: SHORTNAME, Type: object
Column: ORDER_OBS, Type: object
Column: HL7_FIELD_SUBFIELD_ID, Type: object
Column: EXTERNAL_COPYRIGHT_NOTICE, Type: object
Column: EXAMPLE_UNITS, Type: object
Column: LONG_COMMON_NAME, Type: object
Column: EXAMPLE_UCUM_UNITS, Type: object
Column: STATUS_REA

#### Data Distribution Overview

Let's inspect the names, data types, and the top 5 most common values for each column in the DataFrame to get a quick overview of the data distribution.

#### Cardinality Checks for Class, Component, System

In [5]:
#### Cardinality Check:

for col in ["CLASS", "COMPONENT", "SYSTEM"]:
    n_unique = df[col].nunique(dropna=True)
    n_null = df[col].isna().sum()
    print(f"{col}: {n_unique:,} unique values, {n_null:,} nulls (of {len(df):,} total rows)")

CLASS: 440 unique values, 0 nulls (of 112,405 total rows)
COMPONENT: 55,630 unique values, 0 nulls (of 112,405 total rows)
SYSTEM: 2,631 unique values, 0 nulls (of 112,405 total rows)


##### Most Components are Document Ontology components,not clinical/health items.

In [7]:
#### Breakdown Classtype

df["CLASSTYPE"].value_counts()

,count
CLASSTYPE,
1,69651
2,28786
4,12807
3,1161


In [8]:
#### Print data types (all strings)
print(df.dtypes)

LOINC_NUM                    object
COMPONENT                    object
PROPERTY                     object
TIME_ASPCT                   object
SYSTEM                       object
SCALE_TYP                    object
METHOD_TYP                   object
CLASS                        object
VersionLastChanged           object
CHNG_TYPE                    object
DefinitionDescription        object
STATUS                       object
CONSUMER_NAME                object
CLASSTYPE                    object
FORMULA                      object
EXMPL_ANSWERS                object
SURVEY_QUEST_TEXT            object
SURVEY_QUEST_SRC             object
UNITSREQUIRED                object
RELATEDNAMES2                object
SHORTNAME                    object
ORDER_OBS                    object
HL7_FIELD_SUBFIELD_ID        object
EXTERNAL_COPYRIGHT_NOTICE    object
EXAMPLE_UNITS                object
LONG_COMMON_NAME             object
EXAMPLE_UCUM_UNITS           object
STATUS_REASON               

In [14]:
results = []

# Get unique CLASSTYPE values
for classtype_val in df['CLASSTYPE'].unique():
    # Filter DataFrame for the current CLASSTYPE
    filtered_df = df[df['CLASSTYPE'] == classtype_val]

    # Get value counts for 'CLASS' within this CLASSTYPE
    class_counts = filtered_df['CLASS'].value_counts()

    if not class_counts.empty:
        # Get the top 10 classes and their counts
        top_10_classes = class_counts.head(10)

        # Calculate the count for 'Other' classes
        other_count = class_counts.iloc[10:].sum() if len(class_counts) > 10 else 0

        # Store all classes (top 10 + other) for this CLASSTYPE
        current_classtype_classes = []
        for class_name, count in top_10_classes.items():
            current_classtype_classes.append({'CLASS': class_name, 'Count': count})

        if other_count > 0:
            current_classtype_classes.append({'CLASS': 'Other', 'Count': other_count})

        # Calculate total count for this CLASSTYPE
        total_classtype_count = sum(item['Count'] for item in current_classtype_classes)

        # Add to final results with percentage
        for item in current_classtype_classes:
            results.append({
                'CLASSTYPE': classtype_val,
                'CLASS': item['CLASS'],
                'Count': item['Count'],
                'Percentage': (item['Count'] / total_classtype_count * 100) if total_classtype_count > 0 else 0
            })

    else:
        results.append({
            'CLASSTYPE': classtype_val,
            'CLASS': 'No classes found',
            'Count': 0,
            'Percentage': 0
        })

# Create a DataFrame from the results
class_breakdown_df = pd.DataFrame(results)

display(class_breakdown_df)

,CLASSTYPE,CLASS,Count,Percentage
0,2,RAD,7446,25.866741
1,2,DOC.ONTOLOGY,3829,13.301605
2,2,PHENX,3683,12.794414
3,2,H&P.HX,902,3.133468
4,2,PULM,878,3.050094
5,2,CARD.US,817,2.838185
6,2,OB.US,793,2.754811
7,2,PANEL.PHENX,703,2.442159
8,2,H&P.PX,507,1.761273
9,2,CLIN,503,1.747377
